In [55]:
import pandas as pd
import numpy as np
import os
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression
from matplotlib.backends.backend_pdf import PdfPages

In [4]:
OUT_BASE = "/biostats_share/hillandr/data/WW_Mobility_2026_04_20/"
os.makedirs(os.path.dirname(OUT_BASE), exist_ok=True)

In [7]:
daily_data = pd.read_csv(os.path.join(OUT_BASE, "processed/2026_04_20_daily_visits.csv"), parse_dates=["DAY"])

In [48]:
def detrend_group(group: pd.DataFrame):
    # Make sure we are sorted small->large
    tmp_group = group.sort_values("DAY")
    # Compute a temporary variable to use for trend fitting
    tmp_group["tmp_x"] = (tmp_group["DAY"] - tmp_group["DAY"].iloc[0]).dt.days
    # Fit linear trend line
    lg = LinearRegression()
    lg.fit(tmp_group[["tmp_x"]], tmp_group["STOPS_BY_DAY_L"])
    # Residuals
    tmp_group["STOPS_BY_DAY_RESID"] = tmp_group["STOPS_BY_DAY_L"] - lg.predict(tmp_group[["tmp_x"]])
    # Differences
    tmp_group["STOPS_BY_DAY_DIFF"] = tmp_group["STOPS_BY_DAY_L"].diff()
    return tmp_group
    

In [60]:
detrended_daily_data = daily_data.groupby("DESTINATION_AREA_SEWERSHED").apply(detrend_group).reset_index(level=1, drop=True).fillna(0)

In [63]:
with PdfPages("detrend_report.pdf") as pdf:
    for sewershed_name, group_df in detrended_daily_data.groupby("DESTINATION_AREA_SEWERSHED"):
        fig, ax = plt.subplots(figsize=(18,6), ncols=3)
        fig.suptitle(sewershed_name)
        for ax_i, col in zip(ax, ["STOPS_BY_DAY_L", "STOPS_BY_DAY_RESID", "STOPS_BY_DAY_DIFF"]):
            ax_i.plot(group_df["DAY"], group_df[col])
            ax_i.set_title(col)
            lg = LinearRegression()
            lg.fit(group_df[["tmp_x"]], group_df[col])
            ax_i.plot(group_df["DAY"], lg.predict(group_df[["tmp_x"]]))
            
        pdf.savefig(fig)
        plt.close(fig)
        